## Improving the ML pipeline in baseline models

#### Importing Libraries

In [4]:
import pandas as pd
import numpy as np
import category_encoders as ce

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

#### Load data

In [5]:
train = pd.read_csv("./../data/processed/adult_train.csv")
test = pd.read_csv("./../data/processed/adult_test.csv")

train.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### Data Preprocessing

#### Cleaning missing values

In [6]:
train.replace("?", np.nan, inplace=True)
test.replace("?", np.nan, inplace=True)

train.dropna(inplace=True)
test.dropna(inplace=True)


#### Encoding target

In [7]:
train["income"] = train["income"].str.strip().map({"<=50K": 0, ">50K": 1})
test["income"] = test["income"].str.strip().map({"<=50K": 0, ">50K": 1})

#### Encoding Features

In [ ]:
#High-cardinality - Target Encoding
target_enc_cols = ["occupation", "native-country"]
te = ce.TargetEncoder(cols=target_enc_cols)
train[target_enc_cols] = te.fit_transform(train[target_enc_cols], train["income"])
test[target_enc_cols] = te.transform(test[target_enc_cols])

#Low-cardinality - One-Hot Encoding
ohe_cols = ["workclass", "marital-status", "relationship", "race", "sex"]
train = pd.get_dummies(train, columns=ohe_cols)
test = pd.get_dummies(test, columns=ohe_cols)

#Align columns so train/test have the same OHE columns
train, test = train.align(test, join="left", axis=1, fill_value=0)

print(f"Train shape after encoding: {train.shape}")
print(f"Test shape after encoding:  {test.shape}")

Train shape after encoding: (30162, 37)
Test shape after encoding:  (15060, 37)


### Feature Engineering

In [ ]:
for df in [train, test]:
    #Bin age into groups
    df["age_group"] = pd.cut(
        df["age"],
        bins=[0, 25, 45, 65, 100],
        labels=["young", "mid", "senior", "elder"]
    ).cat.codes  #encode as integers

    #Log1p transform for skewed capital columns
    df["capital-gain"] = np.log1p(df["capital-gain"])
    df["capital-loss"] = np.log1p(df["capital-loss"])

    #New feature - Net capital
    df["capital-net"] = df["capital-gain"] - df["capital-loss"]

print("New features: age_group, log1p(capital-gain), log1p(capital-loss), capital-net")

New features: age_group, log1p(capital-gain), log1p(capital-loss), capital-net


### Drop redundant features

In [ ]:
#Verifying education vs education-num correlation before dropping
from sklearn.preprocessing import LabelEncoder
_edu_encoded = LabelEncoder().fit_transform(train["education"])
corr = np.corrcoef(_edu_encoded, train["education-num"])[0, 1]
print(f"Pearson correlation between education (label-encoded) and education-num: {corr:.4f}")


drop_cols = ["education", "fnlwgt"]
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)
print(f"Dropped columns: {drop_cols}")

Pearson correlation between education (label-encoded) and education-num: 0.3454
Dropped columns: ['education', 'fnlwgt']


### Split X and y

In [ ]:
X_train = train.drop("income", axis=1)
X_test = test.drop("income", axis=1)

y_train = train["income"]
y_test = test["income"]

print(f"X_train shape:{X_train.shape}, X_test shape: {X_test.shape}")

X_train shape: (30162, 36), X_test shape: (15060, 36)


### Scale numerical features

In [ ]:
num_cols = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week", "capital-net", "occupation", "native-country", "age_group"]

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

## GBDT — Baseline vs Improved Pipeline

In [13]:
gbdt = GradientBoostingClassifier(n_estimators=100, random_state=42)
gbdt.fit(X_train, y_train)

y_pred = gbdt.predict(X_test)
y_proba = gbdt.predict_proba(X_test)[:, 1]

baseline = {"Accuracy": 0.8611, "F1": 0.6812, "AUC": 0.9181}
improved = {
    "Accuracy": round(accuracy_score(y_test, y_pred), 4),
    "F1":       round(f1_score(y_test, y_pred), 4),
    "AUC":      round(roc_auc_score(y_test, y_proba), 4),
}

comparison = pd.DataFrame({"Baseline GBDT": baseline, "Improved GBDT": improved})
print(comparison)

for metric in ["Accuracy", "F1", "AUC"]:
    delta = improved[metric] - baseline[metric]
    direction = "▲" if delta >= 0 else "▼"
    print(f"{metric}: {direction} {abs(delta):.4f}")

          Baseline GBDT  Improved GBDT
Accuracy         0.8611         0.8647
F1               0.6812         0.6956
AUC              0.9181         0.9191
Accuracy: ▲ 0.0036
F1: ▲ 0.0144
AUC: ▲ 0.0010
